<a href="https://colab.research.google.com/github/yoenisa18-ctrl/Projet-pedagogique-EURO-NOK/blob/main/EURO/NOK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance
!pip install pandas_datareader

In [2]:
import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
import statsmodels.api as sm

In [16]:
START, END = "2023-01-01", "2026-09-09"

brent  = web.DataReader("DCOILBRENTEU", "fred", START, END)
eurnok = yf.download("EURNOK=X", start=START, end=END, auto_adjust=True)["Close"].squeeze()
dxy    = yf.download("DX-Y.NYB", start=START, end=END, auto_adjust=True)["Close"].squeeze()
print(brent.columns)
print(len(brent), len(eurnok), len(dxy))

df = pd.concat([
    brent.rename(columns={"DCOILBRENTEU": "brent"}),
    eurnok.rename("eurnok"),
    dxy.rename("dxy"),
], axis=1).dropna()
print(len(df), df.index.min(), df.index.max())
df.head()

rets = df.pct_change().dropna()
rets.head()

ref  = rets.loc["2023-01-01":"2026-01-31"]
test = rets.loc["2026-02-28":"2026-09-08"]

print("référence :", len(ref), "obs")
print("test      :", len(test), "obs")


X_ref = sm.add_constant(ref[["brent", "dxy"]])
y_ref = ref["eurnok"]

modele = sm.OLS(y_ref, X_ref).fit(cov_type="HC3")
print(modele.summary())

X_test = sm.add_constant(test[["brent", "dxy"]])
y_test = test["eurnok"]

predit = modele.predict(X_test)
residus = y_test - predit

print("résidu moyen   :", residus.mean())
print("erreur-type    :", residus.std() / len(residus)**0.5)
print("ratio          :", residus.mean() / (residus.std() / len(residus)**0.5))
print("part positive  :", (residus > 0).mean())

def lire(m, nom=""):
    t = pd.DataFrame({
        "coef": m.params,
        "err-type": m.bse,
        "p-value": m.pvalues,
        "IC bas": m.conf_int()[0],
        "IC haut": m.conf_int()[1],
    }).round(4)
    print(f"--- {nom} | R² = {m.rsquared:.4f} | n = {int(m.nobs)} ---")
    return t

X2 = sm.add_constant(test[["brent", "dxy"]])
m2 = sm.OLS(test["eurnok"], X2).fit(cov_type="HC3")
lire(m2, "test fév-sept 2026")

b_ref,  ic_ref  = modele.params["brent"], modele.conf_int().loc["brent"]
b_test, ic_test = m2.params["brent"],     m2.conf_int().loc["brent"]

print(f"bêta référence : {b_ref:.4f}   IC95 [{ic_ref[0]:.4f} ; {ic_ref[1]:.4f}]")
print(f"bêta test      : {b_test:.4f}   IC95 [{ic_test[0]:.4f} ; {ic_test[1]:.4f}]")
print()
print("hors IC de référence :", not (ic_ref[0] <= b_test <= ic_ref[1]))

print(m2.params)
print(m2.conf_int())

no = pd.read_csv("GOVT_GENERIC_RATES.csv")
print(no.head())
print(no.dtypes)

no = pd.read_csv("GOVT_GENERIC_RATES.csv", sep=";", parse_dates=["TIME_PERIOD"])
print(no[["TIME_PERIOD", "OBS_VALUE"]].head())
print(no["TIME_PERIOD"].min(), no["TIME_PERIOD"].max(), len(no))

no3y = no.set_index("TIME_PERIOD")["OBS_VALUE"].rename("no3y")

de= pd.read_csv("euroyield.csv", sep=";")
print(de.columns.tolist())
print(de.head())

de = pd.read_csv("euroyield.csv", parse_dates=["DATE"])
de3y = de.set_index("DATE").iloc[:, -1].rename("de3y")
print(de3y.tail(), de3y.dtype)

spread = (no3y - de3y).rename("spread")

df2 = pd.concat([df, spread], axis=1).dropna()

rets2 = pd.DataFrame({
    "eurnok": df2["eurnok"].pct_change(),
    "brent":  df2["brent"].pct_change(),
    "dxy":    df2["dxy"].pct_change(),
    "spread": df2["spread"].diff(),
}).dropna()

ref2  = rets2.loc["2023-01-01":"2026-01-31"]
test2 = rets2.loc["2026-02-28":"2026-09-08"]
print(len(ref2), len(test2))

X_ref2 = sm.add_constant(ref2[["brent", "dxy", "spread"]])
m_ref2 = sm.OLS(ref2["eurnok"], X_ref2).fit(cov_type="HC3")
lire(m_ref2, "référence — modèle complet")

X_t2 = sm.add_constant(test2[["brent", "dxy", "spread"]])
res2 = test2["eurnok"] - m_ref2.predict(X_t2)

print("résidu moyen :", res2.mean())
print("ratio        :", res2.mean() / (res2.std() / len(res2)**0.5))
print("part positive:", (res2 > 0).mean())

m_test2 = sm.OLS(test2["eurnok"], X_t2).fit(cov_type="HC3")
lire(m_test2, "test — modèle complet")



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Index(['DCOILBRENTEU'], dtype='object')
957 957 925
904 2023-01-03 00:00:00 2026-09-01 00:00:00
référence : 759 obs
test      : 125 obs
                            OLS Regression Results                            
Dep. Variable:                 eurnok   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     4.384
Date:                Thu, 10 Sep 2026   Prob (F-statistic):             0.0128
Time:                        12:35:41   Log-Likelihood:                 2919.1
No. Observations:                 759   AIC:                            -5832.
Df Residuals:                     756   BIC:                            -5818.
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.

,coef,err-type,p-value,IC bas,IC haut
const,-0.0003,0.0004,0.3767,-0.0010,0.0004
brent,-0.0055,0.0074,0.4593,-0.0199,0.0090
dxy,-0.0265,0.1306,0.8392,-0.2825,0.2295
spread,-0.0171,0.0144,0.2374,-0.0453,0.0112
